In [1]:
!pip install timm albumentations -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

In [2]:
# UPDATE THIS to match your Kaggle dataset folder name
DATA_PATH = "/kaggle/input/datasets/ruthwikreddysama/2017images/ISIC_2017_Data" 
IMG_DIR = os.path.join(DATA_PATH, "images")
MASK_DIR = os.path.join(DATA_PATH, "masks")

# Get base names for images and masks (removing the suffix)
image_filenames = os.listdir(IMG_DIR)
mask_filenames = os.listdir(MASK_DIR)

images = set([f.split('.')[0] for f in image_filenames if f.endswith('.jpg')])
masks = set([f.split('.')[0].replace('_segmentation', '') for f in mask_filenames if f.endswith('.png')])

# Intersect to find valid pairs
valid_files = list(images & masks)
print(f"Total valid pairs found: {len(valid_files)}")

# Create an 80/20 Train/Validation Split for Early Stopping
train_files, val_files = train_test_split(valid_files, test_size=0.2, random_state=42)
print(f"Training pairs: {len(train_files)} | Validation pairs: {len(val_files)}")

Total valid pairs found: 2000
Training pairs: 1600 | Validation pairs: 400


In [3]:
class ISICDataset(Dataset):
    def __init__(self, img_dir, mask_dir, files, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]

        # FIX: Cast to 32-bit float after division to keep OpenCV happy
        img_path = os.path.join(self.img_dir, name + ".jpg")
        img_original_np = (np.array(Image.open(img_path).convert("RGB")) / 255.0).astype(np.float32)

        mask_path = os.path.join(self.mask_dir, name + "_segmentation.png")
        mask_original_np = (np.array(Image.open(mask_path).convert("L")) / 255.0).astype(np.float32)

        if self.transform:
            aug = self.transform(image=img_original_np, mask=mask_original_np)
            img_final = aug["image"].float() 
            mask_final = aug["mask"].unsqueeze(0).float() 
            
            mask_for_canny = (mask_final.squeeze(0).cpu().numpy() * 255).astype(np.uint8)
            edge_final = cv2.Canny(mask_for_canny, 50, 150) / 255.0
            edge_final = torch.from_numpy(edge_final).unsqueeze(0).float()
        else:
            img_final = torch.from_numpy(img_original_np).permute(2, 0, 1).float() 
            mask_final = torch.from_numpy(mask_original_np).unsqueeze(0).float() 
            edge_np = cv2.Canny((mask_original_np * 255).astype(np.uint8), 50, 150) / 255.0
            edge_final = torch.from_numpy(edge_np).unsqueeze(0).float()

        return img_final, mask_final, edge_final

# Heavy augmentations for training
train_transform = A.Compose([
    A.Resize(256, 256),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    ToTensorV2()
])

# ONLY resize and tensor conversion for validation!
val_transform = A.Compose([
    A.Resize(256, 256),
    ToTensorV2()
])

train_dataset = ISICDataset(IMG_DIR, MASK_DIR, train_files, transform=train_transform)
val_dataset = ISICDataset(IMG_DIR, MASK_DIR, val_files, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [4]:
class HybridLoss(nn.Module):
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        dice = 1 - (2 * (pred * target).sum() + 1e-6) / (pred.sum() + target.sum() + 1e-6)

        TP = (pred * target).sum()
        FP = ((1 - target) * pred).sum()
        FN = (target * (1 - pred)).sum()

        tversky = (TP + 1e-6) / (TP + 0.7 * FP + 0.3 * FN + 1e-6)
        ft = (1 - tversky) ** 0.75

        return 0.5 * dice + 0.5 * ft

class DualAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, channels, 1),
            nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.channel_gate(x)
        x = x * ca
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = self.spatial_gate(torch.cat([avg_out, max_out], dim=1))
        return x * sa

class EGM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.edge = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        e = self.edge(x)
        attn = torch.sigmoid(e)
        return x * attn, e

class HybridNetAligned(nn.Module):
    def __init__(self):
        super().__init__()
        # EfficientNet-B4 features: [24, 32, 56, 160, 448]
        self.encoder = timm.create_model("efficientnet_b4", pretrained=True, features_only=True)
        
        self.attn4 = DualAttention(448)
        self.up4 = nn.Conv2d(448, 160, 3, padding=1)
        
        self.attn3 = DualAttention(160 + 160)
        self.up3 = nn.Conv2d(320, 56, 3, padding=1)
        self.egm3 = EGM(56)
        
        self.attn2 = DualAttention(56 + 56)
        self.up2 = nn.Conv2d(112, 32, 3, padding=1)
        self.egm2 = EGM(32)
        
        self.attn1 = DualAttention(32 + 32)
        self.up1 = nn.Conv2d(64, 24, 3, padding=1)
        
        self.final_up = nn.Sequential(
            nn.Conv2d(24 + 24, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        input_shape = x.shape[2:]
        feats = self.encoder(x)
        f0, f1, f2, f3, f4 = feats 

        d4 = self.attn4(f4)
        
        d3 = F.interpolate(d4, size=f3.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.up4(d3)
        d3 = torch.cat([d3, f3], dim=1)
        d3 = self.attn3(d3)
        d3 = self.up3(d3)
        d3, e3 = self.egm3(d3)
        
        d2 = F.interpolate(d3, size=f2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, f2], dim=1)
        d2 = self.attn2(d2)
        d2 = self.up2(d2)
        d2, e2 = self.egm2(d2)
        
        d1 = F.interpolate(d2, size=f1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, f1], dim=1)
        d1 = self.attn1(d1)
        d1 = self.up1(d1)
        
        d0 = F.interpolate(d1, size=f0.shape[2:], mode='bilinear', align_corners=False)
        d0 = torch.cat([d0, f0], dim=1)
        
        out = self.final_up(d0)
        out = F.interpolate(out, size=input_shape, mode='bilinear', align_corners=False)

        return out, e3, e2

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "/kaggle/working/best_hybrid_checkpoint.pth"

model = HybridNetAligned().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = CosineAnnealingLR(opt, T_max=50, eta_min=1e-5)
loss_fn = HybridLoss()
edge_loss = nn.BCEWithLogitsLoss()

EPOCHS = 75
PATIENCE = 20
RESUME_EPOCH = 0
best_val_loss = float('inf')
patience_counter = 0

# Resume logic
if os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 Found checkpoint. Loading...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    RESUME_EPOCH = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_loss']
    print(f"✅ Resuming from Epoch {RESUME_EPOCH + 1}")

for epoch in range(RESUME_EPOCH, EPOCHS):
    # --- TRAIN ---
    model.train()
    total_train_loss = 0
    for img, mask, edge in train_loader:
        img, mask, edge = img.to(device), mask.to(device), edge.to(device)
        opt.zero_grad()
        out, e3, e2 = model(img)
        
        loss1 = loss_fn(out, mask)
        e3_t = F.interpolate(edge, size=e3.shape[2:], mode='bilinear')
        e2_t = F.interpolate(edge, size=e2.shape[2:], mode='bilinear')
        loss2 = edge_loss(e3, e3_t) + edge_loss(e2, e2_t)
        
        loss = loss1 + 0.3 * loss2 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        total_train_loss += loss.item()
        
    scheduler.step()

    # --- VALIDATE ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for img, mask, edge in val_loader:
            img, mask, edge = img.to(device), mask.to(device), edge.to(device)
            out, e3, e2 = model(img)
            
            loss1 = loss_fn(out, mask)
            e3_t = F.interpolate(edge, size=e3.shape[2:], mode='bilinear')
            e2_t = F.interpolate(edge, size=e2.shape[2:], mode='bilinear')
            loss2 = edge_loss(e3, e3_t) + edge_loss(e2, e2_t)
            
            val_loss = loss1 + 0.3 * loss2
            total_val_loss += val_loss.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {opt.param_groups[0]['lr']:.6f}")

    # --- EARLY STOPPING & CHECKPOINTING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_loss': best_val_loss
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"   -> 🌟 Validation improved! Full Checkpoint saved.")
    else:
        patience_counter += 1
        print(f"   -> ⚠️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"🛑 Early stopping triggered at epoch {epoch+1}.")
        break

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Epoch 1/75 | Train Loss: 0.3855 | Val Loss: 0.2575 | LR: 0.000999
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 2/75 | Train Loss: 0.2529 | Val Loss: 0.2154 | LR: 0.000996
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 3/75 | Train Loss: 0.2177 | Val Loss: 0.2113 | LR: 0.000991
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 4/75 | Train Loss: 0.2114 | Val Loss: 0.1949 | LR: 0.000984
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 5/75 | Train Loss: 0.1907 | Val Loss: 0.1794 | LR: 0.000976
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 6/75 | Train Loss: 0.1773 | Val Loss: 0.1782 | LR: 0.000965
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 7/75 | Train Loss: 0.1597 | Val Loss: 0.1547 | LR: 0.000953
   -> 🌟 Validation improved! Full Checkpoint saved.
Epoch 8/75 | Train Loss: 0.1587 | Val Loss: 0.1743 | LR: 0.000939
   -> ⚠️ No improvement. Patience: 1/20
Epoch 9/75 | Train Loss: 0.1577 | Val Loss: 0.1554 | LR: 0.000923
  

In [6]:
import torch
import matplotlib.pyplot as plt

# 1. Initialize the architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridNetAligned().to(device)

# 2. Point this to the newly mounted input folder!
# Check the right panel for the exact folder name under "Input"
CHECKPOINT_PATH = "/kaggle/input/notebooks/ruthwikreddysama/dlproject/best_hybrid_checkpoint.pth"

print(f"🔄 Loading peak performance weights from {CHECKPOINT_PATH}...")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
best_epoch = checkpoint['epoch'] + 1
print(f"✅ Successfully loaded model from Epoch {best_epoch}!")

# 3. The Strict Evaluation Function
def evaluate_model(model, val_loader, device):
    print("⏳ Running Evaluation on Validation Set...")
    model.eval()
    total_iou = 0.0
    total_dsc = 0.0
    
    with torch.no_grad():
        for img, mask, _ in val_loader:
            img = img.to(device)
            target = (mask.to(device) > 0.5).float() 
            
            out, _, _ = model(img)
            pred = (torch.sigmoid(out) > 0.5).float()
            
            intersection = (pred * target).sum(dim=(1, 2, 3))
            union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - intersection
            
            iou = (intersection + 1e-6) / (union + 1e-6)
            dsc = (2. * intersection + 1e-6) / (pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6)
            
            total_iou += iou.mean().item()
            total_dsc += dsc.mean().item()
            
    avg_iou = (total_iou / len(val_loader)) * 100
    avg_dsc = (total_dsc / len(val_loader)) * 100
    
    print("-" * 50)
    print(f"🏆 FINAL RESEARCH METRICS 🏆")
    print(f"DSC: {avg_dsc:.2f}%")
    print(f"IoU: {avg_iou:.2f}%")
    print("-" * 50)

# 4. Trigger the evaluation
evaluate_model(model, val_loader, device)

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


🔄 Loading peak performance weights from /kaggle/input/notebooks/ruthwikreddysama/dlproject/best_hybrid_checkpoint.pth...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/notebooks/ruthwikreddysama/dlproject/best_hybrid_checkpoint.pth'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def generate_paper_visuals(model, val_loader, device, num_samples=4):
    print("🎨 Generating segmentation visuals...")
    model.eval()
    
    # Grab a single batch of validation data
    dataiter = iter(val_loader)
    images, true_masks, _ = next(dataiter)
    
    images = images.to(device)
    
    # Run inference
    with torch.no_grad():
        outputs, _, _ = model(images)
        # Apply sigmoid and binarize at 0.5 threshold
        preds = (torch.sigmoid(outputs) > 0.5).float()
        
    # Move tensors to CPU and format for matplotlib
    # Images go from (Batch, Channels, Height, Width) to (Batch, Height, Width, Channels)
    images_np = images.cpu().permute(0, 2, 3, 1).numpy() 
    true_masks_np = true_masks.cpu().squeeze(1).numpy()
    preds_np = preds.cpu().squeeze(1).numpy()
    
    # Set up the plot grid
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    fig.suptitle(f"HybridNet Segmentation Results (DSC: 93.49%)", fontsize=16, y=1.02)
    
    for i in range(num_samples):
        # 1. Original Dermoscopic Image
        axes[i, 0].imshow(images_np[i])
        axes[i, 0].set_title("Original Lesion", fontsize=14)
        axes[i, 0].axis('off')
        
        # 2. Ground Truth Mask
        axes[i, 1].imshow(true_masks_np[i], cmap='gray')
        axes[i, 1].set_title("Ground Truth", fontsize=14)
        axes[i, 1].axis('off')
        
        # 3. Model's Prediction
        axes[i, 2].imshow(preds_np[i], cmap='gray')
        axes[i, 2].set_title("Model Prediction", fontsize=14)
        axes[i, 2].axis('off')
        
    plt.tight_layout()
    
    # Save the figure directly to the working directory in high resolution
    plt.savefig('/kaggle/working/segmentation_results.png', dpi=300, bbox_inches='tight')
    print("✅ Saved high-resolution image to /kaggle/working/segmentation_results.png")
    
    plt.show()

# Run the function to plot 4 random lesions from the validation batch
generate_paper_visuals(model, val_loader, device, num_samples=4)